# Optimizers in PyTorch

## Course Structure
1. Introduction (5 mins)
2. Setup and Dataset Loading (15 mins)
3. Understanding SGD (30 mins)
4. Implementing SGD with Momentum (30 mins)
5. Comparing Optimizers (30 mins)
6. Discussion and Summary (10 mins)

## Learning Objectives
- Understand gradient descent optimization fundamentals
- Implement SGD and SGD with Momentum from scratch
- Compare optimizer performance on MNIST
- Gain practical experience tuning optimizers

We're going to walk through some gradient descent optimization algorithms, giving some intuition on how they work and then implementing each one in PyTorch.

## Dataset: MNIST
MNIST is a dataset of hand-drawn digits (0-9), with each digit represented as a 28×28 grayscale image. We will normalize the images and apply data augmentation (random rotation and cropping) to enhance learning.

### **Why Only Training Data?**
We focus on training data since we want to compare optimizer performance in minimizing loss. In real-world applications, lower training loss does not always indicate better generalization.


In [2]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import tqdm

# Set random seed for reproducibility
torch.manual_seed(1234)
random.seed(1234)
np.random.seed(1234)

# Mean and standard deviation for normalization (precomputed)
mean = 0.1307
std = 0.3081

# Define data transformations
train_transforms = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.RandomCrop(28, padding=2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean], std=[std])
])

# Load MNIST dataset
train_data = datasets.MNIST(root='.data', train=True, download=True, transform=train_transforms)
test_data = datasets.MNIST(root='.data', train=False, download=True, transform=train_transforms)

# Define batch size
batch_size = 128

# Create DataLoader for training data
train_dl = data.DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_dl = data.DataLoader(test_data, batch_size=batch_size)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 15.9MB/s]


Extracting .data/MNIST/raw/train-images-idx3-ubyte.gz to .data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 473kB/s]


Extracting .data/MNIST/raw/train-labels-idx1-ubyte.gz to .data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.41MB/s]


Extracting .data/MNIST/raw/t10k-images-idx3-ubyte.gz to .data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 7.98MB/s]

Extracting .data/MNIST/raw/t10k-labels-idx1-ubyte.gz to .data/MNIST/raw



Next, we'll define our architecture: a neural network with a single hidden layer.

The `init_params` function initializes the model parameters using the "Kaiming" (or "He") initialization method. This method is particularly effective when using the ReLU activation function, as it helps maintain a good variance of activations throughout the network layers. The weights are initialized to values drawn from a normal distribution scaled by the number of input units, ensuring that the variance of the activations remains stable. The biases are set to zeros, which is a common practice to ensure that they do not introduce any initial bias in the activations.

ReLU = min(0,x)

Why does Kaiming/He initialization work better for ReLU activation?
___
Kaiming Initialization: $w_{l} \sim \mathcal{N}\left(0,  2/n_{l}\right)$

A gaussian distribution centered over 0 with a standard deviation of $\sqrt{2/{n}_{l}}$, biases initialized to 0.

Where $n_l$ is `fan_in` or the number of incoming connections to the neuron.

Expressed in code as: `weight = np.sqrt(2 / fan_in) * np.random.randn(*weight.shape)`

Unlike other activation functions like sigmoid, ReLU activation outputs 0 for negative input values. This can lead to gradients becoming extremely small or outright disappearing during backpropagation, especially for networks with many layers.

Kaiming initialization scales weights based on the fan-in or incoming connections to each neuron. This ensures that the average activation across layers remains close to 0.5, allowing gradients to flow more effectively and avoiding vanishing.

The scaling factor in Kaiming initialization, $\sqrt{\frac{2}{n_{l}}}$, is chosen carefully to ensure that information isn't lost or amplified excessively between layers, aiding in stable learning.

In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim, hid_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hid_dim)
        self.layer2 = nn.Linear(hid_dim, hid_dim)
        self.layer3 = nn.Linear(hid_dim, output_dim)
        self.init_params()

    def init_params(self):
        for n, p in self.named_parameters():
            if 'weight' in n:
                nn.init.kaiming_normal_(p, nonlinearity='relu')
            elif 'bias' in n:
                nn.init.constant_(p, 0)

    def forward(self, x):
        # x = [batch size, channels, height, width]
        batch_size, *_ = x.shape
        x = x.view(batch_size, -1)
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = self.layer3(x)
        return x

Our model uses a 256-dimensional hidden layer. Again, this is chosen pretty much arbitrarily and smaller values may work just as well.

In [4]:
input_dim = 28 * 28
hid_dim = 256
output_dim = 10

model = MLP(input_dim, hid_dim, output_dim)
model(torch.randn(1, 1, 28, 28)).shape

torch.Size([1, 10])

In supervised learning, where each example belongs to a single class, cross-entropy loss is almost always used. This loss function measures the performance of a classification model whose output is a probability value between 0 and 1. Cross-entropy loss increases as the predicted probability diverges from the actual label, making it an effective measure for classification tasks.

In [5]:
criterion = nn.CrossEntropyLoss()

We'll then put the `.to` method to put the model and the loss function on to our GPU, if we have one.

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
criterion = criterion.to(device)

Next up, we'll define some functions for training the model with our optimizers and plotting the results.

`train_epoch` performs a single epoch of training and returns a list of losses per batch.

In [7]:
def train_epoch(iterator, model, optimizer, criterion, device):
    """Performs one epoch of training."""

    losses = []

    for images, labels in tqdm.tqdm(iterator):
        images = images.to(device) # make sure images are on the same device as model
        labels = labels.to(device) # make sure labels are on the same device as model
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return losses

`train` initializes a model and then performs `n_epochs` of training, storing and returning the loss per batch over all the epochs.

In [8]:
def train(train_dl, model, optimizer, criterion, device, n_epochs=5):
    """Trains the model for the given amount of epochs."""

    losses = []

    model.init_params()

    for _ in range(n_epochs):
        epoch_losses = train_epoch(train_dl, model, optimizer, criterion, device)
        losses.extend(epoch_losses)

    return losses

We have two functions for viewing our results.

`interactive_plot` is used for plotting the results of a single experiment. `plot_losses` plots the results of multiple experiments, which is used to compare optimizers against each other.

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def interactive_plot(losses, labels):
    fig = make_subplots(rows=1, cols=1)
    for loss, label in zip(losses, labels):
        fig.add_trace(go.Scatter(
            y=loss, name=label,
            hovertemplate="Step=%{x}<br>Loss=%{y:.3f}"
        ))
    fig.update_layout(height=500, title="Training Loss Comparison")
    fig.show()

def plot_losses(losses, labels, title=None, ymin=0, ymax=None, figsize=(15,5)):
    """
    Plots multiple loss curves with a legend.

    Args:
        losses: List of loss values for each experiment
        labels: List of labels for each experiment
        title: Title for the plot
        ymin: Minimum y-axis value
        ymax: Maximum y-axis value
        figsize: Size of the figure (width, height)
    """
    fig, ax = plt.subplots(figsize=figsize)

    for loss, label in zip(losses, labels):
        ax.plot(loss, label=label)

    ax.set_title(title)
    ax.set_ylabel('Loss')
    ax.set_xlabel('Update Steps')
    ax.set_ylim(ymin=ymin, ymax=ymax)
    ax.grid()
    ax.legend(loc='upper right') # Add legend in upper right corner

Now let's implement our first optimizer!

## Optimizer 1: Stochastic Gradient Descent (SGD)

Stochastic gradient descent is the simplest optimization algorithm, so it's a good place to start. We take our current model parameters $\theta_t$ and subtract the gradient of those parameters, $\nabla_\theta J(\theta_t)$, multiplied by the "learning rate", $\eta$.

We can think of the learning rate as a parameter that controls the magnitude of the parameter update. If our learning rate is too small then our parameter updates will also be too small for us to train our model in a reasonable amount of time. Conversely, if our learning rate is too large then the size of the parameter updates will be so large that learning will become unstable! If you ever get a `NaN` value for your loss, one of the first things to try would be lowering the learning rate.

The SGD algorithm is:

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta J(\theta_t)$$

However, we don't just have one set of parameters, $\theta$, we have multiple parameters: the weights of layer 1, the biases of layer 1, the weights of layer 2, the biases of layer 2, etc. So we'll subscript the parameters with $i$:

$$\theta_{t+1,i} = \theta_{t,i} - \eta \cdot \nabla_\theta J(\theta_{t,i})$$

We subtract because we want to descend the gradient and move towards a lower loss value. Addition would ascend the gradient, hence it's called gradient ascent.

One final thing to mention is the difference between gradient descent, stochastic gradient descent, mini-batch gradient descent and on-line gradient descent. **Gradient descent** means we calculate the gradient using every single example in our training set and then do a single parameter update. This is relatively slow as in our experiments it means only updating the parameters after seeing all 60,000 examples. The other extreme is **stochastic gradient descent** which means we update our parameters after every single example. This is usually very noisy, so a happy medium is updating the parameters after we have seen a *mini-batch* of examples,  **mini-batch gradient descent**. Lastly, **on-line gradient descent**, which usually implies our model is in production and is being constantly fed new examples on which it is using to update its parameters.

**Gradient descent** is sometimes called **batch gradient descent**, where the whole dataset counts as one giant batch, hence using sampled batch of examples is called a *mini-batch*.

In PyTorch, the optimizer is called *stochastic gradient descent* even though it can do any of the above gradient descent variants. The general rule of thumb is that nowadays when someone mentions stochastic gradient descent then they mean mini-batch gradient descent.

Moving on to the implementation. All optimizers need a way of keeping track of the parameters they're supposed to be updating `model_params` and a learning rate, `lr`. SGD in PyTorch doesn't have a default learning rate but `1e-3` is a common default learning rate value for other optimizers, so we use it here. All optimizers need a `zero_grad` function in order to remove the gradients calculated from the last update step, and a `step` function to perform a parameter update.

Note that any PyTorch method with a trailing underscore, e.g., `.sub_`, means the operation is in-place. This means our `step` function is updating each `param`, a tensor of parameters, in-place. These in-place operations are usually significantly faster non in-place operations.

In [10]:
class SGD:
    def __init__(self, model_params, lr=1e-3):
        self.model_params = list(model_params)
        self.lr = lr

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for param in self.model_params:
            param.sub_(self.lr * param.grad)

optimizer = SGD(model.parameters())
sgd_loss = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 23.37it/s]


Now let's plot it and see what it looks like.

In [11]:
interactive_plot([sgd_loss], ['SGD with lr=1e-3'])

Looks reasonable, the loss starts at a high value as our parameters are randomly initialized and then proceeds to decrease steadily. We can't really tell how "good" it is without comparing it against another optimizer, so let's go ahead and do that now.

## Optimizer 2: SGD with Momentum

One way to think of SGD is a ball rolling down a hill, where areas of high gradient are steep parts of the hill and areas of low gradient are very flat areas. Sometimes the global minima, the point with the lowest loss, is in the middle of a giant flat area. The problem is that because these flat areas have small gradients they also give small update steps which makes learning slow.

We'd want to add something to our optimizer that made it keep the "momentum" gained rolling down the steep hills whilst it's going across the flat areas.

That's exact what SGD with momentum does! Our parameter update is now calculated using a velocity, $v$, which depends on the current gradient multiplied by the learning rate plus the previous velocity multiplied by the momentum $\gamma$.

\begin{align*}
    v_{t,i} &= \gamma \cdot v_{t-1,i} + \eta \cdot \nabla_\theta J(\theta_{t,i})\\
    \theta_{t+1,i} &= \theta_{t,i} - v_{t,i}\\
\end{align*}

If momentum is zero then we don't care about the previous velocity at all and this algorithm becomes SGD. Commonly used momentum values are usually around 0.9.

PyTorch's optimizers are sometimes a little different from the actual algorithms. PyTorch's version of SGD with momentum moves the learning rate outside the equation for velocity:

\begin{align*}
    v_{t,i} &= \gamma \cdot v_{t-1,i} + \nabla_\theta J(\theta_{t,i})\\
    \theta_{t+1,i} &= \theta_{t,i} - \eta \cdot v_{t,i}\\
\end{align*}

If the PyTorch implementation differs then we'll implement the PyTorch version as we use it as a reference.

Note that the velocity `v` is a list of tensors corresponding to the model parameters, so we are storing the velocity of every single parameter in our model.

# Exercises

**Graded question 1:** COMPUTE SGD WITH MOMENTUM AND PLOT THE DIFFERENT CURVES OF 2 HYPERPARAMETERS SETS SEE THE DIFFERENCE.



**Optional Graded question:**
**Create a hyperparameter sweeper and get the best hyperparameters for the given code.**

In [12]:
# Write your code here, write a function to train the model using the SGDMomentum optimizer and ablate over the hyperparamters and choose the best ones according to test accuracy/loss
# return the best hyperparameters and the best model

In [13]:
class SGDMomentum:
    def __init__(self, model_params, lr=1e-3, momentum=0.9):
        self.model_params = list(model_params)
        self.lr = lr
        self.momentum = momentum
        self.velocities = [torch.zeros_like(p) for p in self.model_params]  # Initialize velocities

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for i, param in enumerate(self.model_params):
            self.velocities[i] = self.momentum * self.velocities[i] + param.grad  # Update velocity
            param.sub_(self.lr * self.velocities[i])  # Update parameters

In [14]:
# Assuming you have your model, train_dl, criterion, and device ready

# Create an instance of SGDMomentum optimizer
optimizer = SGDMomentum(model.parameters(), lr=1e-3, momentum=0.9)

# Train your model using the new optimizer
sgd_momentum_loss = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 22.70it/s]


In [15]:

interactive_plot([sgd_momentum_loss], ['SGD with LR = 0.02 Momentum = 0.9'])

In [16]:
# Assuming you have your model, train_dl, criterion, and device ready

# Create an instance of SGDMomentum optimizer
optimizer = SGDMomentum(model.parameters(), lr=1e-3, momentum=0.7)

# Train your model using the new optimizer
sgd_momentum_loss1 = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 23.12it/s]


In [17]:
interactive_plot([sgd_momentum_loss1], ['SGD with lr = 1e-3 Momentum=0.7'])

In [18]:
# Assuming you have sgd_loss and sgd_momentum_loss from previous training

interactive_plot([sgd_loss, sgd_momentum_loss, sgd_momentum_loss1 ], ['SGD with lr=1e-3', 'SGD with lr=1e-3 Momentum=0.9','SGD with lr=1e-2 Momentum=0.7'])

In [19]:
# prompt: # Write your code here, write a function to train the model using the SGDMomentum optimizer and ablate over the hyperparamters and choose the best ones according to test accuracy/loss
# # return the best hyperparameters and the best model

import copy

def train_with_sgd_momentum(train_dl, test_dl, model, criterion, device, learning_rates, momentums, n_epochs=5):
    """
    Trains the model using SGD with Momentum, ablates over hyperparameters, and returns the best ones.

    Args:
        train_dl: Training DataLoader
        test_dl: Testing DataLoader
        model: The model to train
        criterion: The loss function
        device: The device to train on
        learning_rates: A list of learning rates to try
        momentums: A list of momentums to try
        n_epochs: The number of epochs to train for

    Returns:
        A tuple containing the best hyperparameters (learning_rate, momentum) and the best model.
    """
    best_hyperparameters = None
    best_test_accuracy = 0
    best_model = None

    for lr in learning_rates:
        for momentum in momentums:
            model_copy = copy.deepcopy(model) # Crucial: Reset model for each hyperparameter combination
            optimizer = torch.optim.SGD(model_copy.parameters(), lr=lr, momentum=momentum)
            model_copy.init_params() # Reinitialize model parameters

            train_losses = train(train_dl, model_copy, optimizer, criterion, device, n_epochs=n_epochs)

            # Evaluate on test set
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in test_dl:
                    images = images.to(device)
                    labels = labels.to(device)
                    outputs = model_copy(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            test_accuracy = 100 * correct / total

            print(f"Learning Rate: {lr}, Momentum: {momentum}, Test Accuracy: {test_accuracy:.2f}%")

            if test_accuracy > best_test_accuracy:
                best_test_accuracy = test_accuracy
                best_hyperparameters = (lr, momentum)
                best_model = copy.deepcopy(model_copy) # Save a copy of the best model
    return best_hyperparameters, best_model


# Optional Reading

## Optimizer 3: Adagrad (Optional)

One downside with SGD is that we use a single learning rate across all of our parameters, and that this learning rate is fixed through the entirety of training.

Ideally, parameters that are updated more frequently have a lower learning rate and parameters that are updated infrequently have a larger learning rate.

This is what Adagrad does. We use $G_{t,i}$ which is the sum of the squared gradients for parameter $i$ up to, and including, time-step $t$. $G_{t,i}$ is initialized to some value, usually zero by default. As the square of the gradients of a parameter are accumulated, $G_{t,i}$ increases, and thus reduces the learning rate for parameter $i$.

$$\theta_{t+1,i} = \theta_{t,i} - \frac{\eta}{\sqrt{G_{t,i}}+\epsilon} \cdot \nabla_\theta J(\theta_{t,i})$$

where:

$$G_{t,i} = G_{t-1,i} + \Big(\nabla_\theta J(\theta_{t,i})\Big)^2$$

$\epsilon$ is very small number, used to avoid division by zero in the denominator. Sometimes you'll see $\epsilon$ inside the square root, and sometimes it will be outside. PyTorch leaves it outside so we will too.

We implement Adagrad below, initializing $G$ as a list of tensors called `acc_sqr_grads` and using `std` to refer to the denominator of the update step equation.

In [20]:
class Adagrad:
    def __init__(self, model_params, lr=1e-2, init_acc_sqr_grad=0, eps=1e-10):
        self.model_params = list(model_params)
        self.lr = lr
        self.acc_sqr_grads = [torch.full_like(p, init_acc_sqr_grad) for p in self.model_params]
        self.eps = eps

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for param, acc_sqr_grad in zip(self.model_params, self.acc_sqr_grads):
            acc_sqr_grad.add_(param.grad * param.grad)
            std = acc_sqr_grad.sqrt().add(self.eps)
            param.sub_((self.lr / std) * param.grad)

In [21]:
optimizer = Adagrad(model.parameters())

In [22]:
adagrad_loss = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 22.49it/s]


In [28]:
interactive_plot([adagrad_loss], ['ADAgrad with lr = 1e-2'])

In [27]:
plot_loss(adagrad_loss, 'Adagrad with lr=1e-2, init_acc_sqr_grad=0, eps=1e-10')

NameError: name 'plot_loss' is not defined

We can see there's an initial large spike in the loss value. This is due to the initial $G$ values being very small and thus the learning rate is divided by a very small number making it very large. Very large learning rates usually lead to unstable training which give higher loss values.

Let's trim the start to get a better view what the final loss value is.

In [ ]:
plot_loss(adagrad_loss, 'Adagrad with lr=1e-2, init_acc_sqr_grad=0, eps=1e-10', ymax=5.0)

Adagrad beats the other two pretty handily, but that initial spike in loss doesn't look very nice. Maybe if we get rid of that initial spike we can make Adagrad perform even better? Let's try some different initial values for $G$ and store them all in a `adagrad_losses` dictionary. Each key in the dictionary will be the initial $G$ value and the values of the dictionary will be a list of training loss per batch.

In [29]:
adagrad_losses = {0: adagrad_loss}

In [30]:
optimizer = Adagrad(model.parameters(), init_acc_sqr_grad=1.0)

In [31]:
adagrad_losses[1.0] = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 22.89it/s]


In [32]:
optimizer = Adagrad(model.parameters(), init_acc_sqr_grad=0.1)

In [33]:
adagrad_losses[0.1] = train(train_dl, model, optimizer, criterion, device)

 29%|██▉       | 135/469 [00:05<00:14, 23.20it/s]


KeyboardInterrupt: 

In [ ]:
optimizer = Adagrad(model.parameters(), init_acc_sqr_grad=0.01)

In [ ]:
adagrad_losses[0.01] = train(train_dl, model, optimizer, criterion, device)

In [ ]:
optimizer = Adagrad(model.parameters(), init_acc_sqr_grad=0.001)

In [ ]:
adagrad_losses[0.001] = train(train_dl, model, optimizer, criterion, device)

Now let's compare all of our four values for the initial $G$ value.

In [ ]:
labels, losses = zip(*adagrad_losses.items())

plot_losses(losses, labels, 'Adagrad init_acc_sqr_grad Value Comparison', ymax=5.0)

As we can see, performance of Adagrad increases as the initial $G$ value decreases, but decreasing $G$ also increases the initial spike in loss at the beginning of training.

Why does the performance decrease as the initial $G$ value increases? This is the major downside of Adagrad: as $G$ is monotonically increasing at each time-step it will be dividing the learning rate by a monotonically increasing number at each time-step. This causes the size of the steps taken to reduce every update step. As the results for an initial $G$ value of 1.0 show, we can see that these smaller step sizes actually increase the time taken for the model to converge, and in extreme cases will cause the step sizes to approach zero meaning the parameters will stop updating completely.

In practice, we do want the learning rate to decrease whilst training, but ideally would not want it to become zero.

## Optimizer 4: Adadelta (Optional)

All of our update step equations can be written in the form of:

$$\theta_{t+1,i} = \theta_{t,i} + \Delta \theta_{t,i}$$

where $\Delta \theta_{t,i}$ is the size of the parameter update, i.e. in SGD we had:

$$\Delta \theta_{t,i} = - \eta \cdot \nabla_\theta J(\theta_{t,i})$$

and in Adagrad we had:

$$\Delta \theta_{t,i} = - \frac{\eta}{\sqrt{G_{t,i}}+\epsilon} \cdot \nabla_\theta J(\theta_{t,i}))$$

The problem of the Adagrad algorithm was that $G$ was monotonically increasing. Adadelta solves this problem by first taking the Adagrad algorithm and replacing $G_{t,i}$ with $E[g^2]_{t,i}$, an exponential moving average of the square of the gradients so far.

$$E[g^2]_{t,i} = \rho E[g^2]_{t-1,i} + (1-\rho)g^2_{t,i}$$

where $g_{t,i} = \nabla_\theta J(\theta_{t,i})$, which we've done just to simplify the notation, and $\rho$ controls how much we care about the previous gradients in the exponential moving average, $\rho=0$ means we don't care about them at all.

This means our update step equation is:

$$\Delta \theta_t = - \frac{\eta}{\sqrt{E[g^2]_{t,i} + \epsilon}} \cdot g_{t,i}$$

Notice that the $\epsilon$ term has now moved inside the square root, which we're copying from PyTorch.

The problem with the above equation, and in fact all update equations seen so far, is the units of the update do not match the units of the parameters. The updates have units of $\frac{\delta J}{\delta \theta}$, which simplify to $\frac{1}{\text{units of }\theta}$ if we assume the cost function is unitless. However, we want our update equations to have units of $\theta$.

To solve this the Adadelta equation uses a second exponential moving average, but this one is of the parameter updates.

To get the final Adadelta equation, we take our first attempt, but replace $\eta$ with an exponential moving average of the squared parameter updates:

$$E[\Delta \theta^2]_{t-1,i} = \rho E[\Delta \theta^2]_{t-2,i} + (1-\rho)\Delta \theta^2_{t-1,i}$$

Thus, we get:

$$\Delta \theta_{t,i} = - \frac{\sqrt{E[\Delta \theta^2]_{t-1,i} + \epsilon}}{\sqrt{E[g^2]_{t,i}+\epsilon}} \cdot g_{t,i}$$

The units now "match" as we now have units of $\frac{\theta^2}{\theta} = \theta$.

This means that we do not even need to use a learning rate value, however in the PyTorch implementation they do use one (which defaults to 1.0), so they end up with:

$$\Delta \theta_{t,i} = - \eta \cdot \frac{\sqrt{E[\Delta \theta^2]_{t-1,i} + \epsilon}}{\sqrt{E[g^2]_{t,i}+\epsilon}} \cdot g_{t,i}$$

Thus:

$$\theta_{t+1,i} = \theta_{t,i} - \eta \cdot \frac{\sqrt{E[\Delta \theta^2]_{t-1,i} + \epsilon}}{\sqrt{E[g^2]_{t,i}+\epsilon}} \cdot g_{t,i}$$

PyTorch also changes default `eps` value from what it was in Adagrad, from `1e-10` to `1e-6`.

In [ ]:
class Adadelta:
    def __init__(self, model_params, lr=1.0, rho=0.9, eps=1e-6):
        self.model_params = list(model_params)
        self.lr = lr
        self.rho = rho
        self.eps = eps
        self.avg_sqr_grads = [torch.zeros_like(p) for p in self.model_params]
        self.avg_sqr_deltas = [torch.zeros_like(p) for p in self.model_params]

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for param, avg_sqr_grad, avg_sqr_delta in zip(self.model_params, \
                                                      self.avg_sqr_grads, \
                                                      self.avg_sqr_deltas):
            avg_sqr_grad.mul_(self.rho).add_(param.grad * param.grad * (1 - self.rho))
            std = avg_sqr_grad.add(self.eps).sqrt()
            delta = avg_sqr_delta.add(self.eps).sqrt().div(std).mul(param.grad)
            param.sub_(self.lr * delta)
            avg_sqr_delta.mul_(self.rho).add_(delta * delta * (1 - self.rho))

In [ ]:
optimizer = Adadelta(model.parameters())

In [ ]:
adadelta_loss = train(train_dl, model, optimizer, criterion, device)

In [ ]:
plot_loss(adadelta_loss, 'Adadelta with lr=1.0, rho=0.9, eps=1e-6')

We can see that we avoid the large initial spike in loss due to the numerator term (the exponential moving average of parameter updates) starting out very small.

Let's compare Adadelta to all the other algorithms so far.

In [ ]:
# Loss comparision plot here

We can see that Adagrad and Adadelta have pretty much equal performance, but Adadelta doesn't have the large initial spike in loss.

## Optimizer 5: RMSprop (optional)

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_{t,i} + \epsilon}} \cdot g_{t,i}$$

In PyTorch, they move the $\epsilon$ term back outside of the square root. This gives us:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_{t,i}} + \epsilon} \cdot g_{t,i}$$

In the PyTorch implementation they also change the default learning rate to `1e-2`, rename `rho` to `alpha` whilst giving it a new default value of `0.99`, and also change the default `eps` to `1e-8`.

In [24]:
class RMSprop:
    def __init__(self, model_params, lr=1e-2, alpha=0.99, eps=1e-8):
        self.model_params = list(model_params)
        self.lr = lr
        self.alpha = alpha
        self.eps = eps
        self.avg_sqr_grads = [torch.zeros_like(p) for p in self.model_params]

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for param, avg_sqr_grad in zip(self.model_params, self.avg_sqr_grads):
            avg_sqr_grad.mul_(self.alpha).add_(param.grad * param.grad * (1 - self.alpha))
            std = avg_sqr_grad.sqrt().add(self.eps)
            param.sub_((self.lr / std) * param.grad)

In [25]:
optimizer = RMSprop(model.parameters())

In [26]:
rmsprop_loss = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:20<00:00, 22.59it/s]


In [44]:
interactive_plot([rmsprop_loss], ['ADAgrad with lr = 1e-2'])

In [43]:
plot_loss(rmsprop_loss, 'RMSprop with lr=1e-2, alpha=0.99, eps=1e-8')

NameError: name 'plot_loss' is not defined

We run into a similar issue as we did with Adagrad, the small denominator in the initial time-steps lead to large step sizes which give huge spikes in loss values during the early stages of training.

Let's zoom in to get a better view of what's going on.

In [ ]:
plot_loss(rmsprop_loss, 'RMSprop with lr=1e-2, alpha=0.99, eps=1e-8', ymax=5.0)

In [ ]:
rmsprop_losses = {1e-8: rmsprop_loss}

In [ ]:
optimizer = RMSprop(model.parameters(), eps=1e-6)

In [ ]:
rmsprop_losses[1e-6] = train(train_dl, model, optimizer, criterion, device)

In [ ]:
optimizer = RMSprop(model.parameters(), eps=1e-4)

In [ ]:
rmsprop_losses[1e-4] = train(train_dl, model, optimizer, criterion, device)

In [ ]:
optimizer = RMSprop(model.parameters(), eps=1e-2)

In [ ]:
rmsprop_losses[1e-2] = train(train_dl, model, optimizer, criterion, device)

In [ ]:
optimizer = RMSprop(model.parameters(), eps=1)

In [ ]:
rmsprop_losses[1] = train(train_dl, model, optimizer, criterion, device)

In [ ]:
labels, losses = zip(*rmsprop_losses.items())

plot_losses(losses, labels, 'RMSprop eps Value Comparison', ymax=5.0)

Increasing `eps` improves performance to a point, `eps=1e-2` gives the best performance, and then performance starts degrading, `eps=1` gives the worst performance.

Let's compare RMSprop against Adagrad and Adadelta.

In [ ]:
# Loss comparison plot here

It's still a bit hard to tell.

How about we smooth the loss curves with a moving average?

In [ ]:
def moving_average(x, w=5):
    return np.convolve(x, np.ones(w), 'valid') / w

In [ ]:
losses = [adagrad_loss, adadelta_loss, rmsprop_losses[1e-2]]
smoothed_losses = [moving_average(loss) for loss in losses]
labels = ['adagrad', 'adadelta', 'rmsprop']

plot_losses(smoothed_losses, labels, 'Adagrad vs. Adadelta vs. RMSprop', ymax=1.0)

## Adam (Optional)
Adam has an exponential moving average of the gradients, like the momentum term that can be added to SGD, and an exponential moving average of squared gradients, like RMSprop.

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{m_{t,i}}{\sqrt{v_{t,i}}+\epsilon}$$

where:

\begin{align*}
    m_{t,i} &= \beta_1 m_{t-1,i} + (1-\beta_1)g_{t,i} \\
    v_{t,i} &= \beta_2 v_{t-1,i} + (1-\beta_2)g_{t,i}^2
\end{align*}

Adam's $m_{t,i}$ is equal to $v_{t,i}$ from SGD with momentum if it had a $(1-\gamma)$ term. Adam's $v_{t,i} = E[g^2]_{t,i}$ from RMSprop, with $\rho$ replaced by $\beta_2$.

As $m$ and $v$ are initialized to zero and $\beta_1$ and $\beta_2$ are initialized close to one the $m$ and $v$ values calculated on the first few update steps are "biased" towards very small values. This is why we saw a huge loss for the first steps of Adagrad, Adadelta and RMSprop.

To solve this, Adam uses "bias corrected" values of $m$ and $v$, calculated as:

\begin{align*}
    \hat{m}_{t,i} &= \frac{m_{t,i}}{1-\beta_1^t} \\
    \hat{v}_{t,i} &= \frac{v_{t,i}}{1-\beta_2^t}
\end{align*}

This gives the final Adam equation as:

$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_{t,i}}{\sqrt{\hat{v}_{t,i}}+\epsilon}$$

Note that the bias corrected values on the first call to `step` are calculated with $t = 1$ and not $t = 0$.

In [ ]:
class Adam:
    def __init__(self, model_params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.model_params = list(model_params)
        self.lr = lr
        self.beta_1, self.beta_2 = betas
        self.eps = eps
        self.avg_grads = [torch.zeros_like(p) for p in model_params]
        self.avg_sqr_grads = [torch.zeros_like(p) for p in model_params]
        self.n_steps = 0

    def zero_grad(self):
        for param in self.model_params:
            if param.grad is not None:
                param.grad.zero_()

    @torch.no_grad()
    def step(self):
        self.n_steps += 1
        for param, avg_grad, avg_sqr_grad in zip(self.model_params, \
                                                 self.avg_grads, \
                                                 self.avg_sqr_grads):

            avg_grad.mul_(self.beta_1).add_(1 - self.beta_1, param.grad)  # update avg_grad
            avg_sqr_grad.mul_(self.beta_2).addcmul_(1 - self.beta_2, param.grad, param.grad)  # update avg_sqr_grad

            avg_grad_corrected = avg_grad / (1 - self.beta_1 ** self.n_steps)
            avg_sqr_grad_corrected = avg_sqr_grad / (1 - self.beta_2 ** self.n_steps)
            std = avg_sqr_grad_corrected.sqrt().add(self.eps)

            param.sub_(self.lr * avg_grad_corrected / std)  # update parameters

#References
https://ruder.io/optimizing-gradient-descent/

https://mlfromscratch.com/optimizers-explained/#/

https://wiseodd.github.io/techblog/2016/06/22/nn-optimization/

https://pytorch.org/docs/stable/optim.html

https://github.com/pytorch/pytorch/tree/master/torch/optim

https://www.coursera.org/learn/machine-learning/

## Comparing Optimizer Performance
We will now visualize the training loss of different optimizers to compare their effectiveness.

In [ ]:
# Plot training loss for different optimizers
def plot_loss(optimizer_names, loss_values):
    plt.figure(figsize=(10, 5))
    for name, losses in zip(optimizer_names, loss_values):
        plt.plot(losses, label=name)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Optimizer Comparison')
    plt.legend()
    plt.show()


## Summary & Discussion
- SGD is simple but may converge slowly.
- Adam adapts learning rates and often converges faster.
- RMSprop helps stabilize training, especially for non-stationary objectives.

**Discussion Questions:**
1. Why might an optimizer perform better on one dataset than another?
2. How does batch size impact optimization performance?
3. What are some strategies to tune optimizer hyperparameters?

**Questions to Consider:**
1. Why does momentum help convergence?
2. How do learning rates affect training?
3. What are the tradeoffs between optimizers?


In [34]:
class MiniBatchSGD:
    def __init__(self, model_params, lr=1e-3, batch_size=32):
        self.model_params = list(model_params)
        self.lr = lr
        self.batch_size = batch_size  # Added batch_size parameter

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for param in self.model_params:
            param.sub_(self.lr * param.grad / self.batch_size)  # Divide by batch_size

In [37]:
# Assuming you have your model, train_dl, criterion, and device ready

# Create an instance of SGDMomentum optimizer
optimizer =MiniBatchSGD(model.parameters(), lr=1e-3, batch_size=2)

# Train your model using the new optimizer
MiniBatchSGD_loss = train(train_dl, model, optimizer, criterion, device)

100%|██████████| 469/469 [00:21<00:00, 21.99it/s]


In [38]:
interactive_plot([MiniBatchSGD_loss], ['SGD with LR = 0.02 Momentum = 0.9'])

In [39]:
class NAG:
    def __init__(self, model_params, lr=1e-3, momentum=0.9):
        self.model_params = list(model_params)
        self.lr = lr
        self.momentum = momentum
        self.velocities = [torch.zeros_like(p) for p in self.model_params]

    def zero_grad(self):
        for param in self.model_params:
            param.grad = None

    @torch.no_grad()
    def step(self):
        for i, param in enumerate(self.model_params):
            # Lookahead to calculate the gradient at the "looked-ahead" position
            param_lookahead = param - self.momentum * self.velocities[i]

            # Calculate gradient at the lookahead position
            # (You would need to update your training loop to use param_lookahead
            # for calculating loss and gradients)
            # Assuming 'calculate_gradient' is a function that updates param_lookahead.grad
            # calculate_gradient(param_lookahead)

            self.velocities[i] = self.momentum * self.velocities[i] + param_lookahead.grad # Update velocity using gradient at lookahead
            param.sub_(self.lr * self.velocities[i])

In [42]:
# Assuming you have your model, train_dl, criterion, and device ready

# Create an instance of SGDMomentum optimizer
optimizers =NAG(model.parameters(), lr=1e-3, momentum=0.9)

# Train your model using the new optimizer
NAG_loss = train(train_dl, model, optimizers, criterion, device)

  0%|          | 0/469 [00:00<?, ?it/s]


TypeError: unsupported operand type(s) for +: 'Tensor' and 'NoneType'